# Notebook overview: R8_1-CO_Weekly_adjacency_matrix.ipynb

## What this notebook does
Builds the Colorado third-place co-location network pipeline. It turns daily Cuebiq stop records into daily user-user co-presence JSON files, aggregates daily ties into monthly POI-weighted dyadic JSONs, converts monthly JSONs into NetworkX graphs, and computes/inspects network summaries and centralities for pre- and post-disaster months.

## Inputs used
- Cuebiq/Spectus daily stop files: ../Data/stop_df_perday_CO/daily_agg_to_weekly_Stops/{phase}/{week}/stop_df_{YYYYMMDD}
- Cleaned third-place POI polygons: ../Data/stop_df_perday_CO/POIs/CO_pois_TP.csv
- Home CBG files under ../Data/stop_df_perday_CO/home/{phase}/freq_home_*
- Month/week date map for Oct2021, Nov2021, Jan2022, Feb2022
- Python helpers: R11_colocation_function.py, R_monthly_poi_aggregation.py, build_monthly_graphs_universal.py

## Outputs created
- Daily adjacency JSON: ../Data/stop_df_perday_CO/daily_agg_to_weekly_Stops/{phase}/{week}/for_AM_{YYYYMMDD}_R11.json
- Monthly POI-weighted co-location JSON: ../Data/stop_df_perday_CO/daily_agg_to_weekly_Stops/{phase}/{prefix}_{month}/..._POI_weighted_colocation_R11.json
- Monthly NetworkX graph pickles: ../Data/stop_df_perday_CO/graphs_POI_weighted/*_graph_POI_weighted_R11.pkl
- Centrality/summary tables and graph diagnostics used by downstream random-removal, behavior-informed, and bonding/bridging notebooks.

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install seaborn

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
print("Current working directory:", os.getcwd())

In [ ]:
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)

In [ ]:
HOME_FILE_MAP = {
    # -------- PRE-DISASTER --------
    ("pre_disaster", "Oct_2021"):
        "freq_home_20211216_20211229",

    ("pre_disaster", "Nov_2021"):
        "freq_home_20211101_20211130",

    # -------- POST-DISASTER --------
    ("post_disaster", "Jan_2022"):
        "freq_home_20220106_20220203",

    ("post_disaster", "Feb_2022"):
        "freq_home_20220106_20220203",
}

In [ ]:
WEEK_DATE_MAP = {
    # ---------------- POST DISASTER ----------------
    ("post_disaster", "Jan_2022"): ( "2022-01-06", "2022-02-02" ),
    ("post_disaster", "Feb_2022"): ( "2022-02-03", "2022-03-02" ),

    # ---------------- PRE DISASTER  ----------------
    ("pre_disaster", "Oct_2021"): ("2021-10-01", "2021-10-28"),
    ("pre_disaster", "Nov_2021"): ("2021-11-01", "2021-11-28"),
}

In [ ]:
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)


## Function-based R11 co-location workflow

The workflow below uses `run_daily_colocation()` from `R11_colocation_function.py` for every date, then aggregates daily JSON files with `R_monthly_poi_aggregation.py`, and finally builds monthly NetworkX graphs with `build_monthly_graphs_universal.py`.

## Spatial temporal colocation for all 

In [ ]:
def month_dates(year, month):
    """
    Returns list of YYYYMMDD strings for a given month.
    """
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthEnd(1)
    return pd.date_range(start, end, freq="D").strftime("%Y%m%d").tolist()


In [ ]:
import importlib
import R11_colocation_function
importlib.reload(R11_colocation_function)

In [ ]:
from R11_colocation_function import run_daily_colocation

In [ ]:
def run_all_months(revision="R11", R_meters=30):
    phase_weeks = {
        "pre_disaster": ["Oct_2021", "Nov_2021"],
        "post_disaster": ["Jan_2022", "Feb_2022"]
    }

    for Phase, weeks in phase_weeks.items():
        for Week in weeks:
            date_range = WEEK_DATE_MAP.get((Phase, Week))
            if date_range is None:
                raise ValueError(f"No date range defined for Phase={Phase}, Week={Week}")

            start_date, end_date = date_range
            dates = pd.date_range(start=start_date, end=end_date, freq="D").strftime("%Y%m%d")

            print(f"\n▶ {Phase} | {Week} | {len(dates)} days")

            for i, d in enumerate(dates, 1):
                print(f"{i}/{len(dates)} — {d}")

                success = run_daily_colocation(
                    output_date=d,
                    Week=Week,
                    Phase=Phase,
                    revision=revision,
                    stops_dir=stops_dir,
                    pois_dir=pois_dir,
                    base=base,
                    HOME_FILE_MAP=HOME_FILE_MAP,
                    state="CO",
                    R_meters=R_meters
                )

                if not success:
                    print(f"❌ Failed: {Phase} | {Week} | {d}")

In [ ]:
run_all_months(revision="R11")

## check one day before running all

In [ ]:
test_phase = "pre_disaster"
test_week  = "Oct_2021"
test_date  = "20211008"

success = run_daily_colocation(
    output_date=test_date,
    Week=test_week,
    Phase=test_phase,
    revision="R11",
    stops_dir=stops_dir,
    pois_dir=pois_dir,
    base=base,
    HOME_FILE_MAP=HOME_FILE_MAP,
    state="CO",
    R_meters=30
)

print("Success:", success)

In [ ]:
test_path = os.path.join(
    stops_dir,
    f"{test_phase}/{test_week}/for_AM_{test_date}_R11.json"
)

print("Exists:", os.path.exists(test_path))

In [ ]:
with open(test_path) as f:
    data = json.load(f)

print("Number of users with edges:", len(data))

# inspect first 3
for k in list(data.keys())[:3]:
    print(k, "→", data[k])

In [ ]:
def check_symmetry(data):
    for u in data:
        for v in data[u]:
            if v not in data:
                print("Missing reverse user:", v)
                return False
            if u not in data[v]:
                print("Missing reverse edge:", u, "↔", v)
                return False
    return True

print("Symmetric:", check_symmetry(data))

# Monthly Merging and weighting colocation 
for the entire week 
based on frequency of meeting through the week

In [ ]:
import importlib
import R_monthly_poi_aggregation

importlib.reload(R_monthly_poi_aggregation)

from R_monthly_poi_aggregation import build_monthly_colocation

In [ ]:
def run_all_monthly_aggregations(
    WEEK_DATE_MAP,
    stops_dir,
    revision="R8.1"  # or R8.1 / R11 etc
):

    for (phase, week), (window_start, window_end) in WEEK_DATE_MAP.items():

        # Prefix mapping
        if phase == "pre_disaster":
            prefix = "PDM"
        elif phase == "post_disaster":
            prefix = "PtDM"
        else:
            raise ValueError(f"Unknown phase: {phase}")

        # Month tag like Oct2021, Jan2022 etc
        month_tag = week.replace("_", "")

        print("\n" + "="*60)
        print(f"🚀 Processing: {phase} | {week}")
        print(f"📅 Window: {window_start} → {window_end}")
        print("="*60)

        build_monthly_colocation(
            window_start=window_start,
            window_end=window_end,
            week_list=[week],   # single week per month
            stops_dir=stops_dir,
            phase=phase,
            revision=revision,
            prefix=prefix,
            month_tag=month_tag
        )

    print("\n🎯 All monthly aggregations completed.")

In [ ]:
run_all_monthly_aggregations(
    WEEK_DATE_MAP=WEEK_DATE_MAP,
    stops_dir=stops_dir,
    revision="R11"   # or R8.1, R11
)

In [ ]:
# how frequently two users meet? - included as weights - if two users meet 2 time a week weight = 2/7, 
# however if they meet twice in the same day then the weight remain 1/7

# Network Graph - networkX

### funtion for all months at a time


In [ ]:
from build_monthly_graphs_universal import build_monthly_graphs

In [ ]:
from build_monthly_graphs_universal import build_monthly_graphs

In [ ]:
graphs = build_monthly_graphs(
    stops_dir=stops_dir,
    graph_dir=graph_dir,
    revision="R11",      
    save_graphs=True     
)


# Network metrics

In [ ]:
revision = [
    #"R8.1", "R10", "R10_1", "R10_2", 
             "R11", 
    #"R11_1", "R11_2"
]

In [ ]:
def load_nx_graph(pkl_path):
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)
    if not isinstance(G, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        raise TypeError(f"Loaded object is not a NetworkX graph: {type(G)} from {pkl_path}")
    return G

In [ ]:
def ensure_str_nodes(G):
    if any(not isinstance(n, str) for n in G.nodes()):
        return nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}, copy=True)
    return G

In [ ]:
# print(month, os.path.basename(graph_path), G.number_of_nodes(), G.number_of_edges())

## Recerating post graphs with only those users that were in pre also

In [ ]:
def build_graph_files_for_revision(graph_dir, revision):
    return {
        "Oct2021": os.path.join(graph_dir, f"PDM_Oct2021_graph_POI_weighted_{revision}.pkl"),
        "Nov2021": os.path.join(graph_dir, f"PDM_Nov2021_graph_POI_weighted_{revision}.pkl"),
        "Jan2022": os.path.join(graph_dir, f"Jan2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl"),
        "Feb2022": os.path.join(graph_dir, f"Feb2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl"),
    }

In [ ]:
revision = "R11_2"

graph_files = {
    "Oct2021": os.path.join(graph_dir, f"PDM_Oct2021_graph_POI_weighted_{revision}.pkl"),
    "Nov2021": os.path.join(graph_dir, f"PDM_Nov2021_graph_POI_weighted_{revision}.pkl"),
    "Jan2022": os.path.join(graph_dir, f"PtDM_Jan2022_graph_POI_weighted_{revision}.pkl"),
    "Feb2022": os.path.join(graph_dir, f"PtDM_Feb2022_graph_POI_weighted_{revision}.pkl"),
    
}

In [ ]:
pre_months = ["Oct2021", "Nov2021"]

pre_users = set()
for m in pre_months:
    with open(graph_files[m], "rb") as f:
        Gp = pickle.load(f)
    pre_users.update(map(str, Gp.nodes()))

print(f"✅ Pre users (union Oct+Nov): {len(pre_users)}")

In [ ]:
post_months = ["Jan2022", "Feb2022"]

for m in post_months:
    print(f"📦 Building {m}_remaining_pre_users")

    with open(graph_files[m], "rb") as f:
        G_post = pickle.load(f)

    G_post = ensure_str_nodes(G_post)

    # Restrict to users who existed pre
    keep_nodes = set(G_post.nodes()) & pre_users
    G_post_remain_pre = G_post.subgraph(keep_nodes).copy()

    print(
        f"   Nodes: {G_post.number_of_nodes()} → {G_post_remain_pre.number_of_nodes()}\n"
        f"   Edges: {G_post.number_of_edges()} → {G_post_remain_pre.number_of_edges()}"
    )

    out_path = os.path.join(
        graph_dir,
        f"{m}_remaining_pre_users_graph_POI_weighted_{revision}.pkl"
    )

    with open(out_path, "wb") as f:
        pickle.dump(G_post_remain_pre, f)

    print(f"✅ Saved: {out_path}\n")

## Computing Network Metrics and centralities

## Network metrics

In [ ]:
def inspect_graph(G, name="graph"):
    print(f"\n🔍 Inspecting {name}")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())

    if G.number_of_edges() == 0:
        print("⚠️ No edges")
        return

    u, v, d = next(iter(G.edges(data=True)))
    print("Sample edge:", u, v)
    print("Edge attributes:", d.keys())

    assert "weight" in d
    assert "poi_counts" in d

In [ ]:
def summarize_network(H):
    N = H.number_of_nodes()
    E = H.number_of_edges()

    if N == 0:
        return {
            "nodes": 0, "edges": 0, "mean_degree": 0.0,
            "heterogeneity": 0.0, "density": 0.0,
            "norm_density": 0.0
        }

    degs_w = np.array([d for _, d in H.degree(weight="weight")])
    mean_deg = float(degs_w.mean()) if degs_w.size else 0.0
    hetero = float(degs_w.std() / mean_deg) if mean_deg > 0 else 0.0

    raw_density = nx.density(H)
    max_edges = N * (N - 1) / 2
    expected_density = E / max_edges if max_edges > 0 else 0
    norm_density = raw_density / expected_density if expected_density > 0 else 0

    return {
        "nodes": N,
        "edges": E,
        "mean_degree": mean_deg,
        "heterogeneity": hetero,
        "density": raw_density,
        "norm_density": norm_density
    }

In [ ]:
all_rows = []

for revision in revisions:

    print(f"\n📦 Processing revision: {revision}")

    graph_files = build_graph_files_for_revision(graph_dir, revision)

    for month, graph_path in graph_files.items():

        if not os.path.exists(graph_path):
            print(f"❌ Missing: {graph_path}")
            continue

        G = ensure_str_nodes(load_nx_graph(graph_path))
        phase = "pre" if month in ["Oct2021", "Nov2021"] else "post"

        net = summarize_network(G)
        net.update({
            "revision": revision,
            "month": month,
            "phase": phase
        })

        all_rows.append(net)

network_metrics_df = pd.DataFrame(all_rows)
network_metrics_df

In [ ]:
network_metrics_path = os.path.join(
    graph_dir, f"network_metrics_POI_weighted_with_remain_{revision}.csv"
)

network_metrics_df.to_csv(network_metrics_path, index=False)
print("✅ Saved:", network_metrics_path)

## Plot Network basic metrics

In [ ]:
revision = "R11"
network_metrics_path = os.path.join(
    graph_dir, f"network_metrics_POI_weighted_with_remain_all_revisions.csv"
)

network_metrics_df = pd.read_csv(network_metrics_path)


In [ ]:
plot_metrics = ["nodes", "edges", "mean_degree"]

revisions = sorted(network_metrics_df["revision"].unique())

palette = {
    "pre": "black",
    "post": "grey"
}

for revision in revisions:

    df_rev = network_metrics_df.query("revision == @revision")

    fig, axes = plt.subplots(
        1, len(plot_metrics),
        figsize=(5, 2.5),
        sharey=False
    )

    fig.suptitle(f"Revision {revision}", fontsize=13)

    for ax, metric in zip(axes, plot_metrics):

        values = (
            df_rev
            .groupby("phase")[metric]
            .mean()
            .loc[["pre", "post"]]
        )

        phases = ["pre", "post"]
        x = np.arange(len(phases))

        colors = [palette[p] for p in phases]

        ax.bar(x, values.values, color=colors, width=0.6)

        ax.set_title("")
        ax.set_xticks(x)
        ax.set_xticklabels(["Pre", "Post"])
        ax.tick_params(axis="y", labelsize=14)
        ax.tick_params(axis="x", labelsize=14)
        for spine in ax.spines.values():
            spine.set_visible(False)

    plt.tight_layout()
    plt.show()

## Compute centralities

In [ ]:
def compute_node_centralities_contact_fast(G, weight="weight", approx_betweenness_k=500, seed=0):
    # build inv_weight without mutating original edge dicts too much
    inv = {}
    for u, v, d in G.edges(data=True):
        w = d.get(weight, 0.0)
        inv[(u, v)] = (1.0 / w) if (w is not None and w > 0) else 0.0
    nx.set_edge_attributes(G, inv, "inv_weight")

    out = pd.DataFrame(index=list(G.nodes()))
    out["raw_degree"] = pd.Series(dict(G.degree()))
    out["strength"] = pd.Series(dict(G.degree(weight=weight)))

    # closeness using inv_weight as distance
    out["closeness_centrality"] = pd.Series(nx.closeness_centrality(G, distance="inv_weight"))

    # clustering (unweighted, unless you want weighted clustering separately)
    out["clustering_coefficient"] = pd.Series(nx.clustering(G))

    # betweenness: approximate for speed
    if approx_betweenness_k is None:
        out["betweenness_centrality"] = pd.Series(nx.betweenness_centrality(G))
    else:
        out["betweenness_centrality"] = pd.Series(
            nx.betweenness_centrality(G, k=approx_betweenness_k, seed=seed)
        )

    return out

In [ ]:
node_centrality_rows = []
network_centrality_rows = []

PRE_MONTHS = {"Oct2021", "Nov2021"}

for revision in revisions:

    print(f"\n📦 CENTRALITIES for revision: {revision}")

    graph_files = build_graph_files_for_revision(graph_dir, revision)

    for month, graph_path in graph_files.items():

        if not os.path.exists(graph_path):
            print(f"❌ Missing: {graph_path}")
            continue

        print(f"   📖 {month}")

        G = ensure_str_nodes(load_nx_graph(graph_path))
        phase = "pre" if month in PRE_MONTHS else "post"

        # ---- Node-level centralities ----
        node_cent = compute_node_centralities_contact_fast(
            G,
            weight="weight",
            approx_betweenness_k=500,
            seed=0
        )

        node_cent = node_cent.reset_index().rename(columns={"index": "node"})
        node_cent["month"] = month
        node_cent["phase"] = phase
        node_cent["revision"] = revision

        node_centrality_rows.append(node_cent)

        # ---- Network-level summary ----
        row = {
            "revision": revision,
            "month": month,
            "phase": phase,
            "nodes": G.number_of_nodes(),
            "edges": G.number_of_edges(),

            "mean_strength": float(node_cent["strength"].mean()),
            "median_strength": float(node_cent["strength"].median()),

            "mean_closeness": float(node_cent["closeness_centrality"].mean()),
            "median_closeness": float(node_cent["closeness_centrality"].median()),

            "mean_betweenness": float(node_cent["betweenness_centrality"].mean()),
            "median_betweenness": float(node_cent["betweenness_centrality"].median()),

            "mean_clustering": float(node_cent["clustering_coefficient"].mean()),
            "median_clustering": float(node_cent["clustering_coefficient"].median()),
        }

        network_centrality_rows.append(row)

# ---- Combine ----
node_metrics_centrality_df = pd.concat(node_centrality_rows, ignore_index=True)
network_centrality_df = pd.DataFrame(network_centrality_rows)

In [ ]:
node_centrality_path = os.path.join(graph_dir, f"node_metrics_centrality_POI_weighted_{revision}.csv")
network_centrality_path = os.path.join(graph_dir, f"network_centrality_POI_weighted_{revision}.csv")

node_metrics_centrality_df.to_csv(node_centrality_path, index=False)
network_centrality_df.to_csv(network_centrality_path, index=False)

print("✅ Saved node-level centralities:", node_centrality_path)
print("✅ Saved network-level centralities:", network_centrality_path)

In [ ]:
node_metrics_centrality_df

## Plotting Centralities

In [ ]:
node_centrality_path = os.path.join(graph_dir, f"node_metrics_centrality_POI_weighted_all_revisions.csv")
network_centrality_path = os.path.join(graph_dir, f"network_centrality_POI_weighted_all_revisions.csv")

node_metrics_centrality_df = pd.read_csv(node_centrality_path)
network_centrality_df = pd.read_csv(network_centrality_path)

In [ ]:
# Pick what you want to plot (means shown here; you can switch to medians)
plot_metrics = [
    "mean_strength",
    "mean_closeness",
    # "mean_betweenness",
    "mean_clustering",
]

# If you want medians instead, use:
# plot_metrics = ["median_strength","median_closeness","median_betweenness","median_clustering"]

# Keep only the two groups you want to compare
centrality_plot_df = (
    network_centrality_df
    .copy()
    .query("(phase == 'pre') or (phase == 'post')")   # adjust if you later add other populations
    .assign(
        plot_label=lambda df: df["phase"].map({
            "pre": "Pre-Disaster",
            "post": "Post-Disaster (Remaining Pre-Users)"
        })
    )
    # If you later add population column, mirror your earlier logic:
    # .query("(phase == 'pre' and population == 'all') or (phase == 'post' and population == 'remain_pre')")
    .groupby("plot_label")[plot_metrics]
    .mean()
    .loc[["Pre-Disaster", "Post-Disaster (Remaining Pre-Users)"]]
)

labels = centrality_plot_df.index.tolist()
x = np.arange(len(labels))
bar_w = 0.65

palette = {
    "Pre-Disaster": "black",
    "Post-Disaster (Remaining Pre-Users)": "grey"
}

In [ ]:
fig, axes = plt.subplots(
    1, len(plot_metrics),
    figsize=(7.5, 2.7),
    sharey=False
)
plt.subplots_adjust(wspace=1.2)

for ax, metric in zip(axes, plot_metrics):
    heights = centrality_plot_df[metric].values
    colors = [palette[l] for l in labels]

    ax.bar(x, heights, color=colors, width=bar_w, zorder=1)
    ax.set_title(metric.replace("_", " ").title(), fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=90, ha="right")

    # remove box spines (like your earlier plot)
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
for revision in revisions:

    df_rev = df_nodes[df_nodes["revision"] == revision]

    for m in metrics:

        plot_df = df_rev[["Condition", m]].dropna()

        if plot_df.empty:
            continue

        means = plot_df.groupby("Condition")[m].mean()

        pre_mean = means.get("Pre-Disaster", np.nan)
        post_mean = means.get("Post-Disaster (Remaining Pre-Users)", np.nan)

        plt.figure(figsize=(5.2, 4))

        ax = sns.violinplot(
            data=plot_df,
            x="Condition",
            y=m,
            order=order,
            palette=palette,
            inner="box",
            cut=0,
            linewidth=0.6
        )

        # overlay mean dots
        for xpos, cond in enumerate(order):
            if cond in means.index:
                ax.scatter(xpos, means.loc[cond], color="black", s=28, zorder=10)

        # ---- TITLE WITH VALUES ----
        title_metric = metric_map.get(m, m)
        ax.set_title(
            f"{title_metric}\n"
            f"Revision {revision} | "
            f"Pre: {pre_mean:.4f}  |  Post: {post_mean:.4f}",
            fontsize=11
        )

        # ---- X TICKS SIMPLIFIED ----
        ax.set_xticklabels(["pre", "post"])

        ax.set_xlabel("")
        ax.set_ylabel("")

        for spine in ax.spines.values():
            spine.set_visible(False)

        plt.tight_layout()
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

revisions = sorted(node_metrics_centrality_df["revision"].unique())

for revision in revisions:

    df_rev = node_metrics_centrality_df[
        node_metrics_centrality_df["revision"] == revision
    ].copy()

    df_rev["strength"] = pd.to_numeric(df_rev["strength"], errors="coerce")
    df_rev = df_rev.replace([np.inf, -np.inf], np.nan).dropna(subset=["strength"])

    plt.figure(figsize=(6,4))

    sns.histplot(
        data=df_rev,
        x="strength",
        bins=80,
        kde=False
    )

    plt.xlabel("Weighted Degree (Strength)")
    plt.ylabel("Count")
    plt.title(f"Weighted Degree Distribution\nRevision {revision}")
    plt.tight_layout()
    plt.show()

In [ ]:
for revision in revisions:

    df_rev = node_metrics_centrality_df[
        node_metrics_centrality_df["revision"] == revision
    ].copy()

    df_rev["strength"] = pd.to_numeric(df_rev["strength"], errors="coerce")
    df_rev = df_rev[df_rev["strength"] > 0]

    plt.figure(figsize=(6,4))

    sns.histplot(
        data=df_rev,
        x="strength",
        bins=80
    )

    plt.xscale("log")
    plt.yscale("log")

    plt.xlabel("Weighted Degree (log)")
    plt.ylabel("Count (log)")
    plt.title(f"Weighted Degree Distribution (Log-Log)\nRevision {revision}")
    plt.tight_layout()
    plt.show()

## Plot Network

In [ ]:
revision = "R11"

# Pre graphs
pre_graph_files = {
    "Oct2021": os.path.join(graph_dir, f"PDM_Oct2021_graph_POI_weighted_{revision}.pkl"),
    "Nov2021": os.path.join(graph_dir, f"PDM_Nov2021_graph_POI_weighted_{revision}.pkl"),
}

post_months = ["Jan2022", "Feb2022"]  # remaining-pre-users graphs exist for these


def load_nx_graph(pkl_path):
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)
    if not isinstance(G, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        raise TypeError(f"Loaded object is not a NetworkX graph: {type(G)} from {pkl_path}")
    return G


pre_graphs = {m: load_nx_graph(p) for m, p in pre_graph_files.items()}

post_remaining_graphs = {}
for month in post_months:
    post_remain_path = os.path.join(
        graph_dir,
        f"{month}_remaining_pre_users_graph_POI_weighted_{revision}.pkl"
    )
    post_remaining_graphs[month] = load_nx_graph(post_remain_path)



In [ ]:
def largest_component(G):
    if G.number_of_nodes() == 0:
        return G.copy()
    H = G.to_undirected()
    comp = max(nx.connected_components(H), key=len)
    return G.subgraph(comp).copy()

def fast_layout(G, seed=0, method="spring", k=None, iterations=40):
    if method == "spectral":
        return nx.spectral_layout(G)
    if method == "kamada_kawai":
        return nx.kamada_kawai_layout(G)
    return nx.spring_layout(G, seed=seed, k=k, iterations=iterations)

def edge_widths_from_attr(G, weight="weight", min_w=0.3, max_w=4.0, transform="log1p"):
    w = []
    for _, _, d in G.edges(data=True):
        val = d.get(weight, 1.0)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            val = 1.0
        w.append(float(val))

    w = np.asarray(w, dtype=float)
    if w.size == 0:
        return []

    if transform == "log1p":
        w_t = np.log1p(np.maximum(w, 0))
    elif transform == "sqrt":
        w_t = np.sqrt(np.maximum(w, 0))
    else:
        w_t = w.copy()

    w_min, w_max = np.nanmin(w_t), np.nanmax(w_t)
    if not np.isfinite(w_min) or not np.isfinite(w_max) or np.isclose(w_min, w_max):
        return [0.8] * len(w)

    scaled = (w_t - w_min) / (w_max - w_min)
    return (min_w + scaled * (max_w - min_w)).tolist()

def plot_pre_post_no_intersection(
    G_pre,
    G_post,
    *,
    weight="weight",
    layout_method="spring",
    seed=0,
    iterations=40,
    node_size=8,
    node_alpha=0.9,
    edge_alpha=0.20,
    min_edge_width=0.2,
    max_edge_width=5.0,
    width_transform="log1p",
    figsize=(16, 7),
    title_pre="Pre",
    title_post="Post",
    place_new_post_nodes="jitter",  # {"center","jitter"}
    jitter_scale=1e-3,
):
    # take largest component separately (so both are non-empty & readable)
    Gp = largest_component(G_pre)
    Gq = largest_component(G_post)

    # layout on PRE
    pos_pre = fast_layout(Gp, seed=seed, method=layout_method, iterations=iterations)

    # build POST positions using PRE positions where possible
    rng = np.random.default_rng(seed)
    pos_post = {}
    if len(pos_pre) > 0:
        center = np.mean(np.array(list(pos_pre.values())), axis=0)
    else:
        center = np.array([0.0, 0.0])

    for n in Gq.nodes():
        if n in pos_pre:
            pos_post[n] = pos_pre[n]
        else:
            if place_new_post_nodes == "center":
                pos_post[n] = center
            else:  # jitter
                pos_post[n] = center + rng.normal(scale=jitter_scale, size=2)

    # widths
    pre_w  = edge_widths_from_attr(Gp, weight=weight, min_w=min_edge_width, max_w=max_edge_width, transform=width_transform)
    post_w = edge_widths_from_attr(Gq, weight=weight, min_w=min_edge_width, max_w=max_edge_width, transform=width_transform)

    fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

    ax = axes[0]
    nx.draw_networkx_nodes(
    Gp, pos_pre, ax=ax,
    node_size=4,            # smaller
    node_color="skyblue",      # grey
    alpha=0.25,             # translucent
    linewidths=0            # no borders
)

    nx.draw_networkx_edges(Gp, pos_pre, ax=ax, width=pre_w, alpha=edge_alpha)
    ax.set_title(f"{title_pre}\n|V|={Gp.number_of_nodes()}, |E|={Gp.number_of_edges()}\nedge width: '{weight}'")
    ax.axis("off")

    ax = axes[1]
    nx.draw_networkx_nodes(
    Gq, pos_post, ax=ax,
    node_size=4,
    node_color="salmon",
    alpha=0.5,
    linewidths=0
)
    nx.draw_networkx_edges(Gq, pos_post, ax=ax, width=post_w, alpha=edge_alpha)
    ax.set_title(f"{title_post}\n|V|={Gq.number_of_nodes()}, |E|={Gq.number_of_edges()}\nedge width: '{weight}', font_size = 14")
    ax.axis("off")

    plt.show()

In [ ]:
pre_graphs = {m: load_nx_graph(p) for m, p in pre_graph_files.items()}

post_remaining_graphs = {}
for month in post_months:
    post_remain_path = os.path.join(graph_dir, f"{month}_remaining_pre_users_graph_POI_weighted_{revision}.pkl")
    post_remaining_graphs[month] = load_nx_graph(post_remain_path)

# --- run ---
plot_pre_post_no_intersection(
    pre_graphs["Oct2021"],
    post_remaining_graphs["Jan2022"],
    weight="weight",                 # or "composite_weight"
    layout_method="spring",
    iterations=40,
    width_transform="log1p",
    title_pre="Pre: Oct2021",
    title_post="Post: Jan2022 (remaining pre users)",
)

In [ ]:
pre_graphs = {m: load_nx_graph(p) for m, p in pre_graph_files.items()}

post_remaining_graphs = {}
for month in post_months:
    post_remain_path = os.path.join(graph_dir, f"{month}_remaining_pre_users_graph_POI_weighted_{revision}.pkl")
    post_remaining_graphs[month] = load_nx_graph(post_remain_path)

# --- run ---
plot_pre_post_no_intersection(
    pre_graphs["Nov2021"],
    post_remaining_graphs["Jan2022"],
    weight="weight",                 # or "composite_weight"
    layout_method="spring",
    iterations=40,
    width_transform="log1p",
    title_pre="Pre: Oct2021",
    title_post="Post: Jan2022 (remaining pre users)",
)